# Seaborn Fundamentals
Task 29 - Day 29

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

sns.set_theme(style="whitegrid")

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
df = df.drop(columns=["target"])
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]

df.head()

In [ ]:
print(df.shape)
print(df["species"].value_counts())

## 1. Count plot - categorical counts

In [ ]:
# a categorical column built from petal length so there is something to count per species
df["petal_size"] = pd.cut(df["petal_length"], bins=[0, 2, 5, 10],
                          labels=["Small", "Medium", "Large"], right=False)

plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="species", hue="petal_size")
plt.title("Petal Size Category by Species")
plt.xlabel("Species")
plt.ylabel("Number of flowers")
plt.legend(title="Petal size", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

In [ ]:
pd.crosstab(df["species"], df["petal_size"])

**Interpretation:** Petal size separates setosa completely - all 50 setosa flowers are Small and no other species has a Small petal. Versicolor is almost entirely Medium (48 of 50) and virginica almost entirely Large (44 of 50). Only 8 flowers (2 versicolor, 6 virginica) fall on the wrong side of the Medium/Large boundary, which is where the two species overlap.

## 2. Box plot - comparing distributions

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="species", y="petal_length")
plt.title("Petal Length by Species")
plt.xlabel("Species")
plt.ylabel("Petal length (cm)")
plt.show()

In [ ]:
df.groupby("species")["petal_length"].describe()[["min", "25%", "50%", "75%", "max"]]

**Interpretation:** Median petal length climbs from 1.5 cm (setosa) to 4.35 cm (versicolor) to 5.55 cm (virginica). Setosa's largest value (1.9) is well below versicolor's smallest (3.0), so the two never overlap. Versicolor and virginica do overlap slightly, with versicolor reaching 5.1 and virginica starting at 4.5. Setosa's box is very narrow, but the box plot still flags a few of its values as outliers because the spread is so small.

## 3. Histogram - distribution of one variable

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="sepal_length", bins=12, kde=True)
plt.title("Distribution of Sepal Length")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Count")
plt.show()

In [ ]:
df["sepal_length"].describe()

**Interpretation:** Sepal length ranges from 4.3 to 7.9 cm with a mean of 5.84 and a median of 5.8, so the distribution is close to symmetric with a very slight tail toward larger values. The standard deviation is 0.83. The broad, slightly lumpy shape is partly because three species with different average sepal lengths are mixed together in one histogram.

## 4. Scatter plot - relationship between two variables

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="sepal_length", y="petal_length", hue="species")
plt.title("Sepal Length vs Petal Length")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.show()

In [ ]:
print("Overall correlation:", round(df["sepal_length"].corr(df["petal_length"]), 2))
for species, group in df.groupby("species"):
    print(species, round(group["sepal_length"].corr(group["petal_length"]), 2))

**Interpretation:** Sepal length and petal length rise together, with an overall correlation of 0.87. The points form three visible clusters by species. Within each species the relationship is weaker and varies: 0.27 for setosa, 0.75 for versicolor and 0.86 for virginica. So the strong overall trend comes largely from differences between species, not from a tight relationship inside each one.

## 5. Heatmap - correlation between all numeric variables

In [ ]:
corr = df[["sepal_length", "sepal_width", "petal_length", "petal_width"]].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Between Iris Measurements")
plt.show()

**Interpretation:** Petal length and petal width are almost perfectly linked (0.96), and both correlate strongly with sepal length (0.87 and 0.82). Sepal width is the odd one out: it has almost no relationship with sepal length (-0.12) and a moderate negative relationship with the petal measurements (-0.43 with petal length, -0.37 with petal width).

## 6. Pair plot - every relationship at once

In [ ]:
g = sns.pairplot(df[["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]],
                 hue="species")
g.figure.suptitle("Pairwise Relationships in the Iris Dataset", y=1.02)
plt.show()

**Interpretation:** In every plot that involves a petal measurement, setosa forms its own separate cluster, while versicolor and virginica sit next to each other and touch at their edges. One call to `pairplot()` shows what the earlier single charts showed one at a time: petal measurements separate the species far better than sepal width does.

## Seaborn vs Matplotlib - the same chart both ways

In [ ]:
# Matplotlib: scatter plot coloured by species needs a manual loop
plt.figure(figsize=(7, 5))
for species, group in df.groupby("species"):
    plt.scatter(group["sepal_length"], group["petal_length"], label=species)
plt.title("Sepal Length vs Petal Length (Matplotlib)")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.legend(title="species")
plt.show()

In [ ]:
# Seaborn: one call, colour and legend handled by hue
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="sepal_length", y="petal_length", hue="species")
plt.title("Sepal Length vs Petal Length (Seaborn)")
plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.show()

In [ ]:
# Matplotlib: box plot by group needs the data split into lists first
species_list = list(df["species"].unique())
groups = [df[df["species"] == s]["petal_length"] for s in species_list]

plt.figure(figsize=(7, 4))
plt.boxplot(groups)
plt.xticks([1, 2, 3], species_list)
plt.title("Petal Length by Species (Matplotlib)")
plt.xlabel("Species")
plt.ylabel("Petal length (cm)")
plt.show()

| | Matplotlib | Seaborn |
|---|---|---|
| Level | Low-level, full control over every element | High-level, built on top of Matplotlib |
| Grouping by a column | Manual loop or splitting the data into lists | `hue=` or `x=` handles it in one call |
| Works directly with DataFrames | Needs columns passed in separately | `data=df` plus column names |
| Default look | Plain, needs styling work | Cleaner defaults and themes out of the box |
| Statistical charts (box, violin, heatmap, pair plot) | Possible but long | Single function calls |
| Customising details | Most flexible | Usually done by falling back to Matplotlib (`plt.title`, `plt.xlabel`) |

Seaborn does not replace Matplotlib. Every Seaborn chart is a Matplotlib figure underneath, which is why `plt.title()`, `plt.xlabel()` and `plt.show()` still work on Seaborn plots above.

## Interview questions

**What is Seaborn?**
A Python data visualization library built on top of Matplotlib that provides a high-level interface for statistical charts such as box plots, count plots, heatmaps and pair plots. It works directly with Pandas DataFrames and has attractive default styles, so a chart that takes a loop and several lines in Matplotlib is often a single call.

**How is Seaborn related to Matplotlib?**
Seaborn is built on Matplotlib and draws every chart using it. A Seaborn plot is a normal Matplotlib figure, so Matplotlib functions like `plt.title()`, `plt.xlabel()` and `plt.savefig()` work on it directly. Seaborn adds convenience and statistical features on top; Matplotlib stays the engine and the tool for fine-grained control.

**When is a box plot useful?**
When comparing the distribution of a numeric variable across several groups. It shows the median, the middle 50% of the data (the box), the typical range (the whiskers), and outliers in a compact form, so differences in centre and spread between groups are visible at a glance, as with petal length across the three Iris species above.